# Submission: exp29 ResNet-18 downsampling, no coarse dropout, 5-fold

Card: [exp29_downsample_5fold.md](../../experiments/exp29_downsample_5fold.md)

In [ ]:
import os
import tempfile
from pathlib import Path

import pandas as pd

from core.config import ROOT
from core.cv import Config, CrossValidator
from core.data import read_test
from core.submission import make_submission, submit

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    OVERRIDES = {
        "experiment": "cnn-model-downsample-5fold-smoke",
        "folds": [0],
        "epochs": 1,
        "img_size": 64,
        "n_other": 20,
        "num_workers": 0,
        "retrain": True,
        "log_every_n_steps": 1,
    }
    TEST = read_test().head(64)
    PATH = Path(tempfile.mkdtemp()) / "smoke.csv"
else:
    OVERRIDES = {}
    TEST = None
    PATH = None
CONFIG = Config.from_toml(
    ROOT
    / "sections"
    / "model"
    / "configs"
    / "exp29_downsample_5fold"
    / "resnet18.toml",
    **OVERRIDES
)

## Training


In [ ]:
CrossValidator(CONFIG).run()  # finished folds restored from checkpoints

## Predict


In [ ]:
path = make_submission(CONFIG, test=TEST, path=PATH)
pd.read_csv(path).iloc[:, 1:].mean().round(3)  # predicted positive rate per class

## Upload


In [ ]:
if not SMOKE:
    submit(
        path,
        "exp29 ResNet-18 downsample 0.2, no coarse dropout, 5-fold, OOF best-F1 thresholds",
    )